In [11]:
!pip install nbimporter

In [12]:
import torch
import lab07_starter as S

In [14]:
import json

# Load and execute the vae_checks.ipynb file directly
with open('vae_checks.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        code = ''.join(cell['source'])
        exec(code)

print("vae_checks.ipynb loaded successfully! VAE and vae_loss_fn are now available.")

Check 1 - Closed-form KL: 6.5805 | Monte Carlo KL: 6.5815
Check 2 - Encoder grad norm (with trick): 4.6129 | (detached): 0.0000
Check 3 - VAE loss at beta=1.0 runs successfully: 44.5212
vae_checks.ipynb loaded successfully! VAE and vae_loss_fn are now available.


In [15]:
if __name__ == "__main__":
    Xtr, ytr, Xte, yte = S.load_task()
    net = S.train_feature_net(Xtr, ytr)
    fte = S.get_features(net, Xte)

    betas = [0.1, 0.5, 1.0, 2.0, 4.0]
    seeds = [0, 1, 2]

    for beta in betas:
        print(f"\n=== Beta: {beta} ===")
        for seed in seeds:
            vae = VAE(d_latent=8)
            loss_fn = lambda m, xb, yb: vae_loss_fn(m, xb, yb, beta=beta)
            S.train(vae, loss_fn, Xtr, epochs=200, seed=seed)
            
            with torch.no_grad():
                mu_train, log_var_train = vae.encode(Xtr)
                # Active units: variance of posterior mean > 0.01
                active_units = (mu_train.var(dim=0) > 0.01).sum().item()
                
                # Test metrics
                recon_logits, mu_te, log_var_te = vae(Xte)
                recon_term = torch.nn.functional.binary_cross_entropy_with_logits(
                    recon_logits, Xte, reduction='none'
                ).sum(dim=1).mean().item()
                kl_term = (-0.5 * torch.sum(1 + log_var_te - mu_te.pow(2) - log_var_te.exp(), dim=1)).mean().item()
                
                samples = torch.sigmoid(vae.decode(torch.randn(len(Xte), 8)))
                scores = S.score_samples(samples, fte, net, Xtr)
                
            print(f"Seed {seed} | Active Units: {active_units} | Recon: {recon_term:.2f} | KL: {kl_term:.2f} | FD: {scores['frechet']:.2f}")
            if beta == 4.0 and seed == 0:
                S.save_grid(samples[:30], "collapsed_vae.png", ncols=10, title=f"Collapsed VAE (beta={beta})")


=== Beta: 0.1 ===
Seed 0 | Active Units: 8 | Recon: 17.08 | KL: 14.30 | FD: 11.72
Seed 1 | Active Units: 8 | Recon: 17.16 | KL: 14.19 | FD: 10.79
Seed 2 | Active Units: 8 | Recon: 17.17 | KL: 14.24 | FD: 11.14

=== Beta: 0.5 ===
Seed 0 | Active Units: 7 | Recon: 19.12 | KL: 6.06 | FD: 8.05
Seed 1 | Active Units: 7 | Recon: 19.20 | KL: 6.01 | FD: 9.16
Seed 2 | Active Units: 7 | Recon: 19.17 | KL: 6.08 | FD: 11.17

=== Beta: 1.0 ===
Seed 0 | Active Units: 5 | Recon: 20.91 | KL: 3.33 | FD: 12.07
Seed 1 | Active Units: 5 | Recon: 20.95 | KL: 3.49 | FD: 14.09
Seed 2 | Active Units: 5 | Recon: 20.99 | KL: 3.43 | FD: 13.09

=== Beta: 2.0 ===
Seed 0 | Active Units: 3 | Recon: 24.37 | KL: 1.26 | FD: 44.53
Seed 1 | Active Units: 3 | Recon: 24.55 | KL: 1.16 | FD: 45.18
Seed 2 | Active Units: 3 | Recon: 24.03 | KL: 1.19 | FD: 43.53

=== Beta: 4.0 ===
Seed 0 | Active Units: 0 | Recon: 27.25 | KL: 0.00 | FD: 226.74
Seed 1 | Active Units: 0 | Recon: 27.24 | KL: 0.00 | FD: 217.51
Seed 2 | Active Unit